# Simulating with tabulated refractive indices (torch backend)

How to look up a material by name from the tables shipped with `meent` and run a grating
simulation with it.

The tables live under `meent/nk_data/`, split by where they came from:

| folder | source | wavelength unit |
| --- | --- | --- |
| `refractiveindex_info/` | [refractiveindex.info](https://refractiveindex.info) (CC0) | **m** |
| `jLab/` | the lab's MATLAB `optprop_*` library | **m** |
| `filmetrics/`, `matlab/` | tables that predate those | **nm** |

**The folder does not affect lookup.** `read_material_table()` walks the whole tree and uses the
**file name as the key**, case-insensitively.

See `meent/nk_data/README.md` for the full details.

## 0. Importing meent

**If meent is installed (`pip install meent`), skip the next cell** -- `import meent` already
works. Run it only to use a checkout that has not been installed, which is what this notebook
assumes.

In [ ]:
import os
import sys

# Repository root. This notebook sits one level below it, so the parent directory is the root.
# Edit this line if you move the notebook. e.g. MEENT_ROOT = r'E:\Taesang Yun\meent'
MEENT_ROOT = os.path.dirname(os.getcwd())

# Put it first so this checkout wins over any pip-installed meent.
sys.path.insert(0, MEENT_ROOT)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import meent

BACKEND = 2   # 0=numpy, 1=jax, 2=torch

# Confirm the import resolved to the copy you meant to use
print('meent    :', os.path.dirname(meent.__file__))
print('version  :', meent.__version__)

table = meent.read_material_table()
print(f'materials available: {len(table)}')

## 1. Seeing what is available

`meent.print_materials()` lists every table with its wavelength range and reference.

**Check the wavelength range.** Outside it, interpolation pins to the endpoint value (section 7),
so an out-of-range query returns a plausible number rather than failing.

In [ ]:
meent.print_materials()

One material usually has several datasets, differing in the paper they were measured in, the
sample, and the temperature. **The material name alone does not identify the data**, which is why
files are named `<material>_<source>`.

In [ ]:
for material in meent.list_materials():
    if material['name'].startswith('SI_'):
        print(f"{material['name']:22} "
              f"{material['wl_min']:.3e} - {material['wl_max']:.3e} m   "
              f"{material['source'][:60]}")

## 2. Looking up a single index

`meent.find_nk_index(name, table, wavelength)` returns the complex refractive index.

**Wavelength is in metres.** meent itself is unit-agnostic -- only the ratio of wavelength to
period matters -- but these tables are the one place an absolute unit is fixed. Keep
`wavelength`, `period` and `thickness` all in metres.

Note that the imaginary part of an absorbing material comes back **negative**: the solver runs on
the `n - ik` convention, and the lookup returns that sign directly (see section 6).

In [ ]:
print('Si   @633nm  :', meent.find_nk_index('si_aspnes', table, 633e-9))
print('SiO2 @1.55um :', meent.find_nk_index('sio2_malitson', table, 1.55e-6))
print('Au   @1.55um :', meent.find_nk_index('au_johnson', table, 1.55e-6))

# A __real suffix drops k and returns n alone, as a real number.
print('Si   @633nm (n only):', meent.find_nk_index('si_aspnes__real', table, 633e-9))

## 3. Dispersion curves

How n and k vary with wavelength.

In [ ]:
wavelength_grid = np.linspace(400e-9, 1400e-9, 300)
names = ['si_green-2008', 'sio2_malitson', 'au_johnson']

figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name in names:
    nk = np.array([meent.find_nk_index(name, table, w) for w in wavelength_grid])
    axes[0].plot(wavelength_grid * 1e9, nk.real, label=name)
    axes[1].plot(wavelength_grid * 1e9, np.abs(nk.imag), label=name)

axes[0].set_ylabel('n')
axes[1].set_ylabel('|k|')
axes[1].set_yscale('log')
for axis in axes:
    axis.set_xlabel('wavelength (nm)')
    axis.legend(fontsize=8)
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Building a grating and solving

Draw the pattern in `ucell` as **integers**, then substitute materials for them. Each integer
indexes `mat_list`: `0` is the first material, `1` the second.

The value `put_refractive_index_in_ucell` returns is used **as is** -- no `.conj()` or other
correction is needed.

In [ ]:
wavelength = 1.0e-6                      # 1 um
pattern = np.array([[[0, 0, 0, 1, 1, 1, 1, 0, 0, 0]]])   # 0=SiO2, 1=Si

mee = meent.call_mee(
    backend=BACKEND,
    wavelength=wavelength,
    period=[1.0e-6],                     # same unit as wavelength (m)
    fto=[15, 0],                         # Fourier orders
    thickness=[3.0e-7],
    pol=0,                               # 0=TE, 1=TM
)

mee.ucell = mee.put_refractive_index_in_ucell(
    pattern, ['sio2_malitson', 'si_green-2008'], wavelength)

print('ucell type    :', type(mee.ucell).__name__, mee.ucell.dtype)
print('ucell indices :', np.round(np.unique(mee.ucell.numpy()), 4))

result = mee.conv_solve()
reflectance = float(torch.sum(result.de_ri))
transmittance = float(torch.sum(result.de_ti))
print(f'R = {reflectance:.4f}')
print(f'T = {transmittance:.4f}')
print(f'R + T = {reflectance + transmittance:.4f}'
      f'  (absorption = {1 - reflectance - transmittance:.4f})')

On the torch backend both `ucell` and the results are `torch.Tensor`. Pull scalars out with
`float(torch.sum(...))`.

`R + T` falls slightly short of 1 because silicon absorbs weakly at 1 um (k ~ 5e-4). With
lossless materials only it comes out at exactly 1.

## 5. Wavelength sweep

Refractive index depends on wavelength, so each point needs a fresh solver and a fresh call to
`put_refractive_index_in_ucell`.

In [ ]:
sweep = np.linspace(500e-9, 1400e-9, 60)
spectrum_r, spectrum_t = [], []

for wavelength_value in sweep:
    solver = meent.call_mee(backend=BACKEND, wavelength=wavelength_value, period=[1.0e-6],
                            fto=[15, 0], thickness=[3.0e-7], pol=0)
    solver.ucell = solver.put_refractive_index_in_ucell(
        pattern, ['sio2_malitson', 'si_green-2008'], wavelength_value)
    output = solver.conv_solve()
    spectrum_r.append(float(torch.sum(output.de_ri)))
    spectrum_t.append(float(torch.sum(output.de_ti)))

spectrum_r = np.array(spectrum_r)
spectrum_t = np.array(spectrum_t)

plt.figure(figsize=(7, 3.6))
plt.plot(sweep * 1e9, spectrum_r, label='R')
plt.plot(sweep * 1e9, spectrum_t, label='T')
plt.plot(sweep * 1e9, 1 - spectrum_r - spectrum_t, '--', label='absorption')
plt.xlabel('wavelength (nm)')
plt.ylabel('efficiency')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Absorption climbs towards shorter wavelengths, which is the real behaviour of silicon above its
band gap (~1.1 um).

## 6. Sign convention

The solver runs on the `n - ik` convention. Tables store `k` as the positive extinction
coefficient, and `find_nk_index` applies the sign, so an index used straight from the lookup is
already right. Handed `n + ik` instead, a material **amplifies rather than absorbs** and `R + T`
climbs above 1 -- which is the quickest way to catch a sign mistake.

All three backends return the same value, so results do not depend on which one solves.

In [ ]:
wavelength_metal = 1.55e-6


def total_energy(backend, use_conj):
    solver = meent.call_mee(backend=backend, wavelength=wavelength_metal, period=[1.0e-6],
                            fto=[15, 0], thickness=[5.0e-8], pol=0)
    index = solver.put_refractive_index_in_ucell(
        pattern, ['sio2_malitson', 'au_johnson'], wavelength_metal)
    solver.ucell = index.conj() if use_conj else index
    output = solver.conv_solve()
    total = lambda a: float(torch.sum(a)) if torch.is_tensor(a) else float(np.sum(a).real)
    return total(output.de_ri) + total(output.de_ti)


print('SiO2/Au grating @1.55um,  R + T   (must not exceed 1)')
print(f"  torch  as is     : {total_energy(2, False):.6f}   <- correct")
print(f"  numpy  as is     : {total_energy(0, False):.6f}   <- same value, backends agree")
print(f"  torch  .conj()   : {total_energy(2, True):.6f}   <- wrong sign, unphysical")
print(f"  numpy  .conj()   : {total_energy(0, True):.6f}   <- wrong sign, unphysical")

The two backends agree to every digit, and both stay below 1. Applying `.conj()` flips the sign
back to `n + ik` and pushes the total above 1 on either one.

## 7. Querying outside the tabulated range

Interpolation **returns the nearest endpoint value** outside the table (a clamp). It does not
raise and does not produce `nan`, so a plausible-looking number comes back and the mistake is
easy to miss. Hence the warning.

The usual cause is a wavelength given in the wrong unit: nanometres handed to a table written in
metres land far outside the range, so the endpoint is all you ever get.

In [ ]:
import warnings

au_table = np.asarray(table['AU_JOHNSON']).real
print(f'au_johnson range   : {au_table[0, 0]:.3e} to {au_table[-1, 0]:.3e} m')
print(f'value at upper end : n={au_table[-1, 1]:.4f}  k={au_table[-1, 2]:.4f}')
print()

for query in [1.5e-6, 3e-6, 10e-6]:
    value = meent.find_nk_index('au_johnson', table, query)
    mark = '' if query <= au_table[-1, 0] else '   <- out of range, clamped'
    print(f'  {query:.3e} m -> {value.real:8.4f}{value.imag:+9.4f}j{mark}')

print()
print('Wavelength mistakenly given in nm:')
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    meent.find_nk_index('au_johnson', table, 1550)      # 1550 nm passed straight in
    print(' ', str(caught[0].message))

## Summary

```python
import numpy as np
import torch
import meent

table = meent.read_material_table()    # every table under nk_data/, as a dict
meent.print_materials()                # what is available

mee = meent.call_mee(backend=2, wavelength=wl, period=[p], fto=[15, 0],
                     thickness=[t], pol=0)
mee.ucell = mee.put_refractive_index_in_ucell(pattern, ['sio2_malitson', 'au_johnson'], wl)
result = mee.conv_solve()
R = float(torch.sum(result.de_ri))
T = float(torch.sum(result.de_ti))
```

Worth keeping in mind:

1. **Wavelengths are in metres** -- keep `wavelength`, `period` and `thickness` consistent
2. **No `.conj()` anywhere** -- the lookup already returns `n - ik`, on every backend
3. **Check the wavelength range** with `meent.print_materials()`; a warning almost always means
   the unit is wrong
4. **Choose the dataset deliberately** -- silicon alone has three. The `<material>_<source>` name
   tells you which measurement it is

To add a material, append an entry to the list in `tools/convert_refractiveindex_info.py` (public
database) or `tools/convert_matlab_optprop.py` (MATLAB library) and re-run it.